# Run the complete Bayesian ORCA webpage in Jupyter

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sthsci/Orca/blob/main/notebooks/00_run_the_orca_web_app.ipynb)

**Start here.** Launch the same Dash learning and analysis interface used by the project website inside a Google Colab output cell.

Run the cells from top to bottom. Values collected near the start of each notebook are safe places to experiment. Bayesian SMC fitting is deliberately disabled by default in the analysis notebooks because it can take several minutes; set `RUN_INFERENCE = True` when the data checks and descriptive plots look right.

Use synthetic or approved anonymised data only. Do not upload names, clinical metadata, raw microscopy, or a donor key that could identify participants.


## What this notebook provides

This is the shortest route to the complete interface: Bayesian inference 101, synthetic validation, donor-ignorant event counts, donor-aware event counts, trajectory analysis, result downloads, and the optional workspace screen. The scientific pages work without an account; the separate account service is not started in Colab.

The setup cell clones the public GitHub repository only in Colab and installs its pinned dependencies. A local checkout is reused when this notebook runs from the repository.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys


def find_orca_checkout():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "webapp" / "dashapp.py").is_file():
            return candidate
    return None


IN_COLAB = importlib.util.find_spec("google.colab") is not None
ORCA_ROOT = find_orca_checkout()

if ORCA_ROOT is None and IN_COLAB:
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError("ORCA currently requires a Python 3.12 Colab runtime.")
    ORCA_ROOT = Path("/content/Orca")
    if not (ORCA_ROOT / ".git").is_dir():
        subprocess.check_call(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/sthsci/Orca.git",
                str(ORCA_ROOT),
            ]
        )
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(ORCA_ROOT / "requirements.txt"),
        ]
    )

if ORCA_ROOT is None:
    raise RuntimeError("Run this notebook in Colab or from an ORCA checkout.")

sys.path[:0] = [str(ORCA_ROOT), str(ORCA_ROOT / "src")]
print("ORCA checkout:", ORCA_ROOT)
print("Python:", sys.version.split()[0])


## Check the application before launching it

Building the application confirms that every page and callback can be imported. The route table below is also a compact map of what the webpage covers.


In [ ]:
from webapp.dashapp import PAGES, create_app

app = create_app()
[(page.TITLE, page.PATH) for page in PAGES]


## Launch the webpage inline

Change `RUN_WEB_APP` to `True` and run the next cell. Colab will display the full Dash application below it. Keep the cell running while you use the interface. Computation-heavy SMC fits continue only while the Colab runtime is connected.

The account/CSV-sharing workspace needs the separate platform API and is intentionally not enabled here. Every scientific workflow remains available without it.


In [ ]:
RUN_WEB_APP = False  # change to True in Colab

if RUN_WEB_APP:
    app.run(jupyter_mode="inline", debug=False, port=8050)
else:
    print("Ready. Set RUN_WEB_APP = True to display Bayesian ORCA here.")


## Where to go next

- Use **Bayesian inference 101** for the conceptual primer.
- Use **Event count analysis** when each row is a cell and the outcome is a total count.
- Add donor labels for the **donor-aware** hierarchy.
- Use **Trajectory inference** when each cell has an ordered sequence of unsuccessful (`0`) and successful (`1`) contacts.
- Use the dedicated notebooks in this collection when you prefer editable code and tables over the webpage controls.

Preview settings are for learning and workflow checks. Record the priors, SMC settings, random seed, package version, and input data when producing scientific results.
